In [17]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Incremental_Loading") \
    .getOrCreate()  

In [18]:
df = spark.read.csv("data/orders.csv", header=True, inferSchema=True)

In [19]:
from pyspark.sql.functions import to_date

df = df.withColumn("order_date", to_date("order_date"))

In [20]:
from pyspark.sql.functions import current_date, date_add

last_loaded_date = "2026-03-25"

In [21]:
incremental_df = df.filter(df.order_date > last_loaded_date)

incremental_df.show()

+--------+------+----------+
|order_id|amount|order_date|
+--------+------+----------+
|       3|   150|2026-03-26|
+--------+------+----------+



In [22]:
df = df.withColumn("processing_date", date_add(current_date(), 1))

In [23]:
df.show()

+--------+------+----------+---------------+
|order_id|amount|order_date|processing_date|
+--------+------+----------+---------------+
|       1|   100|2026-03-24|     2026-03-27|
|       2|   200|2026-03-25|     2026-03-27|
|       3|   150|2026-03-26|     2026-03-27|
+--------+------+----------+---------------+



In [24]:
incremental_df.write \
    .mode("append") \
    .partitionBy("processing_date") \
    .parquet("warehouse/orders")

TypeError: 'JavaPackage' object is not callable